# Facial expression comparison (FER2013) -- Colab runner

The Center Loss comparison repeated on a second task: **ResNet-18 with plain
cross-entropy versus ResNet-18 with cross-entropy + Center Loss**, on FER2013.

The room experiments concluded that Center Loss can only organise a
representation that is already adequate. Facial expressions test that where it
should hold most strongly, since Center Loss was proposed for face recognition
in the first place.

Run the cells top to bottom on a **T4 GPU** runtime. Cell 6 does everything:
both variants, both evaluations, and the summary table.

**Before running:** the code in `fer_baseline/` must be pushed to GitHub, or
cell 3b will copy it from Drive instead. Cell 4 checks and tells you which
happened.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and install dependencies

Clones the repository. `fer_baseline/` borrows the training infrastructure from
`vit_s16_baseline/`, so both folders must be present -- cloning the repo gives
you both.

The clone cell deliberately stays at the repository root: `fer_baseline/` may not
be in the clone yet, so changing into it here would fail and leave the working
directory somewhere unexpected. Cell 6 changes into it once it is certainly
present.

In [ ]:
import os

REPO_DIR = '/content/room-classification'
BRANCH   = 'main'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SilviuBR24/room-classification.git {REPO_DIR}
!cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

# Absolute path on purpose: vit_s16_baseline holds the shared requirements and
# is always present in the clone.
!pip install -q -r {REPO_DIR}/vit_s16_baseline/requirements.txt

%cd {REPO_DIR}
print('Code ready on branch:', BRANCH)
print('fer_baseline present in clone:', os.path.isdir(f'{REPO_DIR}/fer_baseline'))

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3b. Fall back to Drive if the code is not in the clone

Only does anything if `fer_baseline/` is missing or incomplete in the clone,
which happens when the folder has not been pushed yet. Copying from Drive is a
stopgap: pushing is better, because then the run is reproducible from the
repository alone.

In [ ]:
import os, shutil

REPO_DIR     = '/content/room-classification'
PROJECT      = f'{REPO_DIR}/fer_baseline'
DRIVE_FOLDER = '/content/drive/MyDrive/Dissertation_Thesis/fer_baseline'

NEEDED = ['config_fer.yaml', 'fer_model.py', 'shared_infrastructure.py',
          'train_fer.py', 'evaluate_fer.py', 'compare_fer.py']

missing = [f for f in NEEDED
           if not os.path.isfile(os.path.join(PROJECT, f))]

if not missing:
    print('fer_baseline is complete in the clone; nothing to do.')
elif os.path.isdir(DRIVE_FOLDER):
    os.makedirs(PROJECT, exist_ok=True)
    for f in os.listdir(DRIVE_FOLDER):
        src = os.path.join(DRIVE_FOLDER, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(PROJECT, f))
    still = [f for f in NEEDED if not os.path.isfile(os.path.join(PROJECT, f))]
    print(f'Copied fer_baseline from Drive (was missing: {missing})')
    if still:
        raise SystemExit(f'Still missing after the copy: {still}')
else:
    raise SystemExit(
        f'fer_baseline is incomplete in the clone (missing: {missing}) and '
        f'{DRIVE_FOLDER} does not exist. Push the folder to GitHub, or upload '
        f'it to that Drive path, then re-run this cell.')

print('Ready:', sorted(f for f in os.listdir(PROJECT) if not f.startswith('_')))

## 4. Unpack the dataset

`fer2013.zip` was produced by `make_fer_zip.py`, which writes entry names with
forward slashes so the tree extracts correctly on Linux. It is copied to the
VM's local SSD first: reading tens of thousands of small files straight from the
Drive FUSE mount is orders of magnitude slower.

In [ ]:
import os, time, shutil, zipfile

DRIVE_ZIP = '/content/drive/MyDrive/Dissertation_Thesis/fer2013.zip'
LOCAL_ZIP = '/content/fer2013.zip'
DATA_ROOT = '/content/fer2013'

EXPECTED = {'train': 28709, 'val': 3589, 'eval': 3589}

if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    if not os.path.exists(LOCAL_ZIP):
        t0 = time.time(); shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
        print(f'Copied zip Drive->local in {time.time()-t0:.0f}s')
    t1 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        z.extractall('/content')
    print(f'Extracted in {time.time()-t1:.0f}s')
else:
    print(f'{DATA_ROOT} already present, skipping.')

ok = True
for split, n in EXPECTED.items():
    p = os.path.join(DATA_ROOT, split)
    counts = {c: len(os.listdir(os.path.join(p, c))) for c in sorted(os.listdir(p))}
    total = sum(counts.values())
    ok &= total == n
    print(f'{split:6s}: {total:6d} images (expected {n})')
    print(f'        {counts}')

assert ok, 'Image counts do not match the published FER2013 split.'
print('\nDataset verified against the published split.')

## 5. Note on the class imbalance

Unlike the room dataset, which had exactly 3,200 training images per class,
FER2013 ranges from 7,215 for *happy* down to 436 for *disgust* -- a ratio of
about 16 to 1.

Overall accuracy alone is therefore misleading: a model that never predicts
*disgust* forfeits only about 1.5% of it. The runner reports macro-averaged F1
alongside accuracy for that reason.

The imbalance also interacts with the implemented Center Loss variant, whose
centre update is normalised by mini-batch size rather than by the number of
samples of that class in the batch. With a batch of 64 the *disgust* centre is
updated far less often than the *happy* centre. On the balanced room dataset
that deviation could not show itself; here it can.

## 6. Run the full comparison (one cell, both variants)

Writes the Colab-specific paths into the config, then runs `compare_fer.py`,
which trains both variants back to back, evaluates each on the held-out test set
(saving embeddings for the geometric analysis) and prints the summary.

Roughly **40 to 50 minutes per variant** on a T4 at 96x96, so about **1.5 hours
total**. Logs stream one clean line per epoch.

Safe to interrupt: results are appended to `fer_comparison_results.csv` after
each variant finishes, and re-running this cell skips whatever already
completed.

In [ ]:
import os, sys, yaml, subprocess

PROJECT   = '/content/room-classification/fer_baseline'
DATA_ROOT = '/content/fer2013'
RUNS_DIR  = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
os.chdir(PROJECT)

cfg = yaml.safe_load(open('config_fer.yaml'))
cfg['data']['train_dir'] = DATA_ROOT + '/train'
cfg['data']['val_dir']   = DATA_ROOT + '/val'
cfg['data']['eval_dir']  = DATA_ROOT + '/eval'
cfg['paths']['output_root']    = RUNS_DIR
cfg['training']['num_workers'] = os.cpu_count()
yaml.safe_dump(cfg, open('config_fer_colab.yaml', 'w'), sort_keys=False)

print('resolution :', cfg['model']['image_size'])
print('epochs     :', cfg['training']['epochs'],
      '| batch', cfg['training']['batch_size'],
      '| workers', cfg['training']['num_workers'])
print('lambda     :', cfg['training']['center_loss_weight'])

env = {**os.environ, 'TQDM_DISABLE': '1', 'PYTHONUNBUFFERED': '1'}
p = subprocess.Popen([sys.executable, '-u', 'compare_fer.py',
                      '--config', 'config_fer_colab.yaml'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
for line in p.stdout:
    sys.stdout.write(line.decode('utf-8', 'replace')); sys.stdout.flush()
print('exit code:', p.wait())

## 7. (Optional) Embedding geometry of the two runs

Runs the same compactness analysis used for the room experiments, on the
embeddings saved in step 6. This is what actually answers the question: whether
Center Loss reorganises the representation on this task, or merely rescales it
as it did on the transformer.

Read the scale-invariant columns -- the inter/intra ratio, the silhouette
coefficient and the Davies-Bouldin index. The absolute distances will shrink
whatever happens, because that is what the objective optimises.

In [ ]:
import glob, os, sys, subprocess

RUNS_DIR = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
ce = sorted(glob.glob(f'{RUNS_DIR}/*_fer_crossentropy'))
cl = sorted(glob.glob(f'{RUNS_DIR}/*_fer_crossentropy_centerloss'))
assert ce and cl, 'run cell 6 first'

env = {**os.environ, 'TQDM_DISABLE': '1', 'PYTHONUNBUFFERED': '1'}
p = subprocess.Popen([sys.executable, '-u',
                      '../vit_s16_baseline/analyze_embeddings.py',
                      '--runs', ce[-1], cl[-1],
                      '--labels', 'FER ResNet-18 (CE)', 'FER ResNet-18 (CE + Center Loss)',
                      '--out-dir', '../figuri/fer_geometry'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
for line in p.stdout:
    sys.stdout.write(line.decode('utf-8', 'replace')); sys.stdout.flush()
p.wait()

## 8. (Optional) How well did the class centres learn?

The implemented Center Loss updates the centres by gradient descent on the
combined objective, normalised by mini-batch size. On this imbalanced dataset
that predicts a measurable effect: the centre of a rare class should track its
true centroid less closely than the centre of a frequent one.

This cell measures it, per class, by comparing each learned centre with the
empirical centroid of that class in the test embeddings. In 512 dimensions two
unrelated vectors have a cosine similarity of about 0.044, which is the
reference for "did not learn".

In [ ]:
import glob, os, sys
import numpy as np, torch

RUNS_DIR = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
CLASSES  = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
TRAIN_N  = dict(zip(CLASSES, [3995, 436, 4097, 7215, 4830, 3171, 4965]))

runs = sorted(glob.glob(f'{RUNS_DIR}/*_fer_crossentropy_centerloss'))
assert runs, 'run cell 6 first'
run = runs[-1]

ck = torch.load(f'{run}/checkpoints/best_model.pt', map_location='cpu', weights_only=False)
state = ck.get('center_loss_state_dict')
assert state, 'this run has no Center Loss state'
C = state['centers'].float().numpy()

emb = sorted(glob.glob(f'{run}/outputs/eval_*/embeddings.npy'))[-1]
X = np.load(emb); y = np.load(emb.replace('embeddings.npy', 'labels.npy'))

print(f'{"class":<10}{"train imgs":>11}{"cosine":>9}')
print('-' * 30)
for k, name in enumerate(CLASSES):
    mu = X[y == k].mean(0)
    cos = float(C[k] @ mu / (np.linalg.norm(C[k]) * np.linalg.norm(mu)))
    print(f'{name:<10}{TRAIN_N[name]:>11,}{cos:>9.3f}')
print('-' * 30)
print(f'random reference in {C.shape[1]} dimensions: {1/np.sqrt(C.shape[1]):.3f}')